<a href="https://colab.research.google.com/github/eojin22/ESAA/blob/main/OB_0928_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **0928 수상작 리뷰**

## 주제 및 데이터
### 주제: 데이콘 Basic 스트레스 지수 예측 : 건강 데이터로 마음의 균형을 찾아라!

**데이터 구성:**

train.csv
- ID : 샘플별 고유 ID
- gender : 성별
- age : 연령
- height : 키(cm)
- weight : 몸무게(kg)
- cholesterol : 콜레스테롤 수치
- systolic_blood_pressure : 수축기 혈압
- diastolic_blood_pressure : 이완기 혈압
- glucose : 혈당 수치
- bone_density : 골밀도
- activity : 생활 운동 강도
- smoke_status : 흡연 상태
- medical_history : 만성질환
- family_medical_history : 가족력
- sleep_pattern : 수면패턴
- edu_level : 학력
- mean_working : 1주일당 평균 근로 시간
- stress_score : 스트레스 점수(Target)

## 코드리뷰
**1. 데이터 전처리 및 파생변수 생성**
- 결측치의 특성을 확인한 후 다른 변수와 연관성이 적은 경우 Unknown 등의 값으로 대체
- 기존 변수만 사용하는 것이 아니라 건강 및 생활습관 데이터의 특성을 반영하여 다양한 파생변수 생성
키와 몸무게를 이용하여 BMI를 계산하고, 근로시간과 수면패턴을 조합하여 work_sleep_risk와 같은 위험 관련 변수를 생성
- 질병, 교육 수준 등의 범주형 변수는 Label Encoding을 적용하여 모델이 사용할 수 있는 형태로 변환

**2. 다양한 변수 조합을 활용한 피처 엔지니어링**
- 기존 피처들의 단순한 값만 사용하는 것이 아니라 변수 간의 관계를 활용하여 새로운 피처를 생성
- 피처 간 곱셈, 나눗셈, 절댓값 차이, 조화평균 등의 연산을 활용
- 트리 기반 모델이 변수 간의 복잡한 관계를 모두 효율적으로 표현하지 못할 수 있기 때문에, 의미 있는 변수 조합을 직접 생성하여 모델에 제공
- 여러 변수의 조합을 통해 스트레스 점수와 관련된 새로운 패턴을 찾고자 함

**3. 피처 중요도 분석 및 SHAP 활용**
- 생성된 모든 피처를 무조건 모델에 사용하는 것이 아니라 SHAP 분석을 통해 각 피처가 예측에 미치는 영향을 확인
- SHAP 값을 기준으로 중요도가 낮은 피처를 제거하고 모델의 성능을 비교
- 특정 임계값을 설정하여 중요도가 높은 피처만 선별함으로써 불필요한 피처를 줄이고 모델의 효율을 높임

**4. CatBoost 모델 적용**
- 여러 모델을 비교한 결과 가장 좋은 성능을 보인 CatBoost Regressor를 최종 모델로 사용
- 건강 데이터에는 범주형 변수와 수치형 변수가 함께 존재하기 때문에 이러한 데이터를 효과적으로 처리할 수 있는 CatBoost를 활용
- Optuna를 이용하여 CatBoost의 하이퍼파라미터를 최적화하고 모델의 성능을 개선

**5. Optuna를 활용한 하이퍼파라미터 최적화**
- 학습률, 트리 깊이, 반복 횟수 등의 하이퍼파라미터를 Optuna를 통해 탐색
- 여러 하이퍼파라미터 조합을 실험하여 검증 데이터에서 가장 좋은 성능을 보이는 조합을 탐색
- 수작업으로 하나씩 값을 변경하는 것보다 효율적으로 최적의 파라미터를 찾을 수 있도록 구성

**6. Seed Ensemble을 통한 성능 개선**
- 하나의 랜덤 시드만 사용하는 것이 아니라 여러 개의 Seed를 사용하여 동일한 모델을 반복 학습
- 각 모델에서 얻은 예측값에 가중치를 적용하여 결합
서로 다른 Seed에서 발생하는 학습 결과의 변동성을 줄이고 보다 안정적인 최종 예측값을 생성
- 이를 통해 과적합을 완화하고 일반화 성능을 높이고자 함

## 코드 흐름 요약
건강 데이터를 전처리하고 다양한 파생변수를 생성한 후 SHAP을 이용해 중요 피처를 선별하고, CatBoost와 Seed Ensemble을 통해 스트레스 점수를 예측하는 과정

  - 데이터 로드 → 결측치 및 범주형 변수 전처리 → 파생변수 생성 → 변수 조합 및 군집 피처 생성 → CatBoost 학습 → SHAP 분석 및 피처 선별 → Optuna 하이퍼파라미터 최적화 → 여러 Seed로 모델 학습 → Seed Ensemble → 최종 스트레스 점수 예측

In [ ]:
# CatBoost 모델 학습
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    verbose=False
)

model.fit(X_train, y_train)

In [ ]:
# SHAP을 활용한 피처 중요도 분석
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)

shap.summary_plot(shap_values, X_train)

In [ ]:
# 여러 Seed를 활용한 Ensemble
predictions = []

for seed in seeds:
    model = CatBoostRegressor(
        random_seed=seed,
        **best_params
    )

    model.fit(X_train, y_train)
    predictions.append(model.predict(X_test))

final_prediction = np.average(
    predictions,
    axis=0,
    weights=weights
)

## 차별점 및 배울 점
**1. 피처 간 다양한 연산을 활용한 피처 엔지니어링**
  - 기존 피처들의 곱, 나눗셈, 절댓값 차이, 조화평균 등을 활용하여 새로운 피처를 생성한 점이 인상적. 특히 트리 기반 모델이 변수 간의 복잡한 수학적 관계를 항상 효율적으로 표현하기는 어렵기 때문에, 의미 있는 변수 조합을 직접 만들어 모델에 제공할 수 있다는 점을 새롭게 알게 됨.

**2. SHAP을 활용한 피처 중요도 분석 및 선별**
- 단순히 많은 피처를 모델에 사용하는 것이 아니라 SHAP을 이용하여 각 피처가 예측 결과에 미치는 영향을 분석하고, 중요도가 낮은 피처를 제거한 점이 인상적.이를 통해 피처를 많이 추가하는 것뿐만 아니라 실제로 중요한 변수를 선별하는 과정도 중요하다는 것을 배움.

**3. Seed Ensemble을 활용한 과적합 완화**
- 하나의 랜덤 시드로 학습한 모델만 사용하는 것이 아니라 여러 Seed를 적용하여 여러 모델을 학습하고 예측 결과를 가중합한 점이 인상적.이를 통해 서로 다른 Seed에서 발생하는 모델의 변동성을 줄이고 보다 안정적인 예측 결과를 얻을 수 있다는 것을 알게 됨.